# User notebook: from images to tracks

This notebook follows a small image sequence through the public object-oriented interface. `HPC` turns each image into observations and prepares an auditable association problem; a `Solver` returns a choice; then `HPC` applies the shared Bayesian and Kalman update before the next image.

`ClassicalSolver` is the implemented solver used to advance the examples. `QuantumSolver` is deliberately a manual input/output adapter with status `not_implemented`: it contains no simulator and this notebook does not present a fabricated quantum answer.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display
import numpy as np

from neutral_atom_mht import ClassicalSolver, HPC, HPCConfig, QuantumSolver

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    raise RuntimeError("Open user_notebook.ipynb from the repository root.")

## A self-contained image sequence

Four small images contain one bright, cell-sized disk moving by two pixels per frame. Keeping this example synthetic makes the notebook executable before the optional microscopy TIFF files are downloaded.

In [ ]:
rows, columns = np.mgrid[:64, :64]
centres = ((18, 20), (20, 21), (22, 22), (24, 23))
images = tuple(
    np.where((columns - x) ** 2 + (rows - y) ** 2 <= 6**2, 255, 0).astype(np.uint8)
    for x, y in centres
)
images[0].shape, images[0].dtype, len(images)

In [ ]:
config = HPCConfig()
hpc = HPC(config, sequence="synthetic")
classical = ClassicalSolver(maximum_nodes=20)
quantum = QuantumSolver()

The first image has no earlier tracks to associate, so one convenience call detects the cell and initializes tracking state.

In [ ]:
first_observed = hpc.observe(images[0], frame=0)
first_step = hpc.step(images[0], classical, frame=0)
[(track.track_id, track.position, track.posterior_probability) for track in first_step.tracks]

## One frame, step by step

For the second image we pause between the objects. `observe()` converts pixels to typed observations. `prepare_frame()` runs the separately callable prediction, gating, weight, filtering, graph-encoding, and clustering stages without changing retained tracks. Its result exposes each intermediate value.

In [ ]:
observed = hpc.observe(images[1], frame=1)
predicted = hpc.predict(frame=1)
gated = hpc.gate(predicted, observed.observations)
calculated = hpc.calculate_weights(predicted, gated)
filtered = hpc.filter_hypotheses(calculated)
graph = hpc.encode_graph(filtered)
clusters = hpc.cluster(graph)
embedding = hpc.graph_embedding(graph)
prepared = hpc.prepare_frame(images[1], frame=1)
assert prepared.predicted_tracks == predicted
assert prepared.gated_associations == gated
assert prepared.hypotheses == filtered
assert prepared.graph == graph
assert prepared.clusters == clusters
{
    "observations": [
        (item.observation_id, item.x, item.y) for item in observed.observations
    ],
    "predicted_tracks": [item.track_id for item in prepared.predicted_tracks],
    "gated_pairs": [
        (item.track_id, item.observation_id) for item in prepared.gated_associations
    ],
    "weights": {item.hypothesis_id: item.weight for item in prepared.hypotheses},
    "graph_fingerprint": prepared.graph.fingerprint,
    "clusters": [item.node_ids for item in clusters],
    "logical_embedding": embedding,
}

A graph node is one possible track-observation association. An edge means that two candidates cannot both be selected because they reuse a track or observation.

In [ ]:
graph_path = hpc.visualize_graph(
    prepared.graph, project_root / "outputs/user_conflict_graph.png"
)
display(Image(filename=str(graph_path)))

The frozen problems now leave `HPC`, pass through the common solver contract, and return as a validated classical run. Only `advance()` below mutates the retained track state.

In [ ]:
classical_run = hpc.solve(prepared, classical)
[item.to_dict() for item in classical_run.results]

## Neutral-atom hand-off boundary

The same immutable solver input can be serialized for a future manual neutral-atom implementation. Calling the placeholder is safe and explicit: it returns `not_implemented`, selects nothing, and must not be used to advance the tracker. `format_output()` is available later to normalize a manually supplied response into the common result contract.

In [ ]:
neutral_atom_input = quantum.format_input(prepared.solver_inputs()[0])
placeholder_run = hpc.solve(prepared, quantum)
{
    "formatted_input": neutral_atom_input.to_dict(),
    "placeholder_status": placeholder_run.results[0].status,
    "successful": placeholder_run.successful,
}

We explicitly choose the successful classical run. `advance()` now applies the common update and mutates the retained state; the unsuccessful placeholder run is never applied.

In [ ]:
advanced = hpc.advance(prepared, classical_run)
[
    (track.track_id, track.position, track.posterior_probability)
    for track in advanced.tracks
]

## A complete multi-frame sequence

A fresh controller can repeat the same image-to-HPC-to-solver-to-HPC exchange for an arbitrary number of frames.

In [ ]:
sequence_hpc = HPC(config, sequence="synthetic")
sequence_history = sequence_hpc.run_sequence(
    images, classical
)
[
    {
        "frame": step.frame,
        "track_positions": [track.position for track in step.tracks],
        "solver_statuses": [item.status for item in step.solver_run.results],
    }
    for step in sequence_history.steps
]

## Optional published benchmark

When the versioned artifacts are present, this final cell reads their summary only. Gold-standard information remains post-hoc evidence and is never passed into `HPC.observe()`.

In [ ]:
summary_path = project_root / "artifacts/summary.json"
artifact_summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else None
artifact_summary["primary_metrics"] if artifact_summary else "Run cell-detect run to create artifacts."